In [1]:
import pandas as pd 
import numpy as np 


In [2]:
temp_df = pd.read_csv("dataset.csv")

In [3]:
df = temp_df[:10000].copy()

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [6]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
print(df.duplicated().sum())

17


In [9]:
df.drop_duplicates(inplace=True)

In [10]:
print(df.duplicated().sum())

0


# basic preprocessing 

In [11]:
import re 
def remove_html_tags(text):
    pattern=re.compile('<.?/>*')
    return pattern.sub(r'',text)

In [12]:
df['review']=df['review'].apply(remove_html_tags)

In [13]:
df['review']= df['review'].str.lower()

In [14]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    
    filtered_words = [
        word for word in words 
        if word.lower() not in stop_words
    ]
    
    return ' '.join(filtered_words)

In [15]:
df['review'] = df['review'].apply(remove_stopwords)

In [16]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. <br /><br />the f...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive
...,...,...
9995,"fun, entertaining movie wwii german spy (julie...",positive
9996,"give break. anyone say ""good hockey movie""? kn...",negative
9997,movie bad movie. watching endless series bad h...,negative
9998,"movie probably made entertain middle school, e...",negative


In [17]:
X = df.iloc[:,0:1]
Y = df['sentiment']

In [18]:
X

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. <br /><br />the f...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [19]:
Y

0       positive
1       positive
2       positive
3       negative
4       positive
          ...   
9995    positive
9996    negative
9997    negative
9998    negative
9999    positive
Name: sentiment, Length: 9983, dtype: object

In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(Y)

In [21]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [23]:
X_train.shape

(7986, 1)

In [24]:
# apply bow 
from sklearn.feature_extraction.text import CountVectorizer

In [25]:
cv = CountVectorizer()

In [26]:
x_train_bow = cv.fit_transform(X_train['review']).toarray()
x_test_bow = cv.transform(X_train['review']).toarray()

In [27]:
x_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

# Train model
gnb.fit(x_train_bow, y_train)

# Prediction
y_pred = gnb.predict(x_test_bow)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))